# Enrich GSS Variable CSVs with Code-to-Answer Mappings

This notebook extracts value labels from GSS Stata (`.dta`) files and adds a `CodeMapping` column
to two variable list CSVs:
- `gss_demographic_variables.csv`
- `gss_politicized_lifestyle_variables.csv`

For each variable, the code mapping shows how numeric response codes map to text answers
(e.g., `1=male; 2=female`). For purely numeric variables (age, hours, etc.), the mapping
indicates `numeric` with any boundary labels noted.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load Stata files and extract code-to-label mappings

In [ ]:
# Paths to GSS Stata data files
STATA_2024 = '/project/jevans/maxzhuyt/gss-depth/GSS2024.dta'
STATA_2022 = '/project/jevans/maxzhuyt/gss-depth/GSS2022.dta'

# Read 2024 (numeric + categorical) - works with convert_categoricals
print('Loading GSS 2024...')
df_num_24 = pd.read_stata(STATA_2024, convert_categoricals=False)
df_cat_24 = pd.read_stata(STATA_2024, convert_categoricals=True)
print(f'  {df_num_24.shape[1]} variables, {df_num_24.shape[0]} observations')

# Read 2022 - use low-level interface due to duplicate category labels in some variables
print('Loading GSS 2022...')
df_num_22 = pd.read_stata(STATA_2022, convert_categoricals=False)
reader_22 = pd.io.stata.StataReader(STATA_2022)
vl_22 = reader_22.value_labels()
var_to_lbl_22 = dict(zip(reader_22._varlist, reader_22._lbllist))
print(f'  {df_num_22.shape[1]} variables, {df_num_22.shape[0]} observations')
print(f'  {len(vl_22)} label sets available')

In [ ]:
def extract_code_mapping(df_num, df_cat, var_name, max_code=100000):
    """
    Extract code-to-label mapping for a variable by comparing
    numeric and categorical versions of the same data.
    
    Returns:
        dict: {code: label} mapping, or None if variable not found
    """
    if var_name not in df_num.columns:
        return None
    
    mapping = {}
    for n, c in zip(df_num[var_name], df_cat[var_name]):
        if pd.notna(n) and pd.notna(c) and int(n) < max_code:
            code = int(n)
            label = str(c).strip()
            # Skip labels that are just the number repeated (truly numeric)
            mapping[code] = label
    
    return dict(sorted(mapping.items())) if mapping else None


def classify_mapping(mapping):
    """
    Classify whether a mapping is categorical, numeric, or mixed.
    
    Returns:
        str: 'categorical', 'numeric', or 'mixed'
    """
    if mapping is None:
        return 'unknown'
    
    numeric_count = 0
    total = len(mapping)
    for code, label in mapping.items():
        # Check if label is just the number (possibly with .0)
        try:
            if float(label) == code:
                numeric_count += 1
        except ValueError:
            pass
    
    if numeric_count == total:
        return 'numeric'
    elif numeric_count > total * 0.7:
        return 'mostly_numeric'
    else:
        return 'categorical'


def format_mapping_string(mapping, var_type):
    """
    Format a code mapping into a concise string for the CSV.
    
    For categorical: "0=never; 1=less than once a year; 2=once or twice a year"
    For numeric: "numeric (18-89, 89=89 or older)"
    For mostly numeric: "numeric (17=17 and younger; 45=45 and older)"
    """
    if mapping is None:
        return ''
    
    if var_type == 'numeric':
        codes = sorted(mapping.keys())
        return f'numeric ({codes[0]}-{codes[-1]})'
    
    if var_type == 'mostly_numeric':
        # Show only the boundary labels that differ from the code
        boundary_labels = []
        for code, label in sorted(mapping.items()):
            try:
                if float(label) != code:
                    boundary_labels.append(f'{code}={label}')
            except ValueError:
                boundary_labels.append(f'{code}={label}')
        
        codes = sorted(mapping.keys())
        if boundary_labels:
            return f'numeric ({codes[0]}-{codes[-1]}; {"; ".join(boundary_labels)})'
        else:
            return f'numeric ({codes[0]}-{codes[-1]})'
    
    # Categorical: show all mappings
    parts = [f'{code}={label}' for code, label in sorted(mapping.items())]
    return '; '.join(parts)

In [ ]:
def extract_code_mapping_22(var_name, max_code=100000):
    """
    Extract code-to-label mapping from 2022 data using low-level StataReader.
    """
    if var_name not in df_num_22.columns:
        return None
    lbl_name = var_to_lbl_22.get(var_name, '')
    if not lbl_name or lbl_name not in vl_22:
        return None
    labels = vl_22[lbl_name]
    mapping = {}
    for code, label in labels.items():
        if int(code) < max_code:
            mapping[int(code)] = str(label).strip()
    return dict(sorted(mapping.items())) if mapping else None


def get_all_mappings(var_names):
    """
    Extract code mappings for a list of variables.
    Tries 2024 data first, falls back to 2022.
    """
    results = {}
    sources = {}
    
    for var in var_names:
        # Try 2024 first (uses numeric vs categorical comparison)
        mapping = extract_code_mapping(df_num_24, df_cat_24, var)
        source = '2024'
        
        # Fall back to 2022 (uses low-level label dictionary)
        if mapping is None:
            mapping = extract_code_mapping_22(var)
            source = '2022'
        
        if mapping is None:
            results[var] = ''
            sources[var] = 'not found'
            continue
        
        var_type = classify_mapping(mapping)
        results[var] = format_mapping_string(mapping, var_type)
        sources[var] = source
    
    return results, sources

## 2. Load variable CSVs

In [ ]:
CSV_DIR = 'gss_question_lists'

demo_df = pd.read_csv(f'{CSV_DIR}/gss_demographic_variables.csv')
lifestyle_df = pd.read_csv(f'{CSV_DIR}/gss_politicized_lifestyle_variables.csv')

print(f'Demographic variables: {len(demo_df)}')
print(f'Politicized lifestyle variables: {len(lifestyle_df)}')

## 3. Extract mappings for all variables

In [ ]:
# Combine all unique variable names
all_vars = list(dict.fromkeys(
    list(demo_df['VariableName']) + list(lifestyle_df['VariableName'])
))
print(f'Total unique variables: {len(all_vars)}')

# Extract mappings
mappings, sources = get_all_mappings(all_vars)

# Summary
found = sum(1 for v in mappings.values() if v)
not_found = [var for var in all_vars if not mappings[var]]
print(f'\nMappings found: {found}/{len(all_vars)}')
print(f'Not found in either Stata file: {len(not_found)}')
if not_found:
    print(f'  Missing: {not_found}')

In [ ]:
# Preview some mappings
for var in ['sex', 'race', 'degree', 'marital', 'attend', 'age', 'educ', 'health', 'relig', 'owngun', 'spanking']:
    if var in mappings:
        print(f'{var} [{sources.get(var, "?")}]: {mappings[var]}')

## 4. Handle missing variables

For variables not found in the Stata files, we check the GSS Data Explorer pages.
Since the site is a JavaScript SPA and can't be scraped directly, we provide known
mappings for common missing variables.

In [ ]:
# Known mappings for variables missing from Stata files
# These are documented in the GSS Codebook
MANUAL_MAPPINGS = {
    'betrlang': '1=English; 2=Spanish/other language; 3=both equally',
    'health1': '1=excellent; 2=very good; 3=good; 4=fair; 5=poor',
    'height': 'numeric (inches, 48-84)',
    'weight': 'numeric (pounds, 80-450)',
    'jew': '1=orthodox; 2=conservative; 3=reform; 4=none of these',
    'jew16': '1=orthodox; 2=conservative; 3=reform; 4=none of these',
    'laidoff': '1=yes; 2=no',
    'oth16': 'denomination code (see codebook)',
    'other': 'denomination code (see codebook)',
    'ratetone': 'skin color scale (1-10, lightest to darkest)',
    'size': 'numeric (place size in 1000s)',
    'spjew': '1=orthodox; 2=conservative; 3=reform; 4=none of these',
    'spother': 'denomination code (see codebook)',
    'sppres80': 'numeric (occupational prestige score, 12-89)',
    'srcbelt': '1=other urban; 2=suburban; 3=other rural; 4=other urban (not SMSA); 5=other rural (not SMSA); 6=not assigned',
    'waypaid': '1=salaried; 2=paid by the hour; 3=other',
    'wealth': 'numeric (total wealth in dollars)',
    'wrksched': '1=day shift; 2=afternoon shift; 3=night shift; 4=rotating shift; 5=split shift; 6=irregular/on-call; 7=other',
    'wrktype': '1=independent contractor; 2=on-call; 3=paid by temp agency; 4=work for contractor; 5=regular permanent employee',
    'yearsjob': 'numeric (years at current job)',
    'emailhr': 'numeric (email hours per week)',
    'emailmin': 'numeric (email minutes per week)',
    'evcrack': '1=yes; 2=no',
    'evidu': '1=yes; 2=no',
    'evpaidsx': '1=yes; 2=no',
    'evstray': '1=yes; 2=no; 3=never married',
    'grndemo': '1=yes; 2=no',
    'grngroup': '1=yes; 2=no',
    'grnmoney': '1=yes; 2=no',
    'grnsign': '1=yes; 2=no',
    'hrsrelax': 'numeric (hours per day to relax)',
    'nobuygrn': '1=always; 2=often; 3=sometimes; 4=never; 5=not available where I shop',
    'partnrs5': 'numeric (number of sex partners in last 5 years, 0-989)',
    'recycle': '1=always; 2=often; 3=sometimes; 4=never',
    'rfamlook': 'numeric (hours per week looking after family)',
    'rhhwork': 'numeric (hours per week on household work)',
    'slpprblm': '1=never; 2=rarely; 3=sometimes; 4=often',
    'stress12': '1=yes; 2=no',
    'usewww': '1=yes; 2=no',
    'wrkhome': '1=daily; 2=several times a week; 3=several times a month; 4=once a month; 5=less often; 6=never',
    'wwwhr': 'numeric (web hours per week)',
    'wwwmin': 'numeric (web minutes per week)',
    'condom': '1=yes; 2=no; 3=did not have sex',
    'cooking1': '1=always me; 2=usually me; 3=about equal/both; 4=usually spouse/partner; 5=always spouse/partner; 6=third person',
    'hhclean1': '1=always me; 2=usually me; 3=about equal/both; 4=usually spouse/partner; 5=always spouse/partner; 6=third person',
    'laundry1': '1=always me; 2=usually me; 3=about equal/both; 4=usually spouse/partner; 5=always spouse/partner; 6=third person',
    'shop1': '1=always me; 2=usually me; 3=about equal/both; 4=usually spouse/partner; 5=always spouse/partner; 6=third person',
    'relexp': '1=yes; 2=no',
    'savesoul': '1=yes; 2=no',
    'esop': '1=yes; 2=no',
}

# Fill in missing mappings
for var, manual_map in MANUAL_MAPPINGS.items():
    if var in mappings and not mappings[var]:
        mappings[var] = manual_map
        sources[var] = 'manual'

# Check what's still missing
still_missing = [var for var in all_vars if not mappings.get(var, '')]
print(f'After manual fill: {len(still_missing)} still missing')
if still_missing:
    print(f'  {still_missing}')

## 5. Add CodeMapping column and save enriched CSVs

In [ ]:
# Enrich demographic variables
demo_df['CodeMapping'] = demo_df['VariableName'].map(mappings)
demo_df.to_csv(f'{CSV_DIR}/gss_demographic_variables.csv', index=False)
print(f'Saved: {CSV_DIR}/gss_demographic_variables.csv')

filled = demo_df['CodeMapping'].notna() & (demo_df['CodeMapping'] != '')
print(f'  {filled.sum()}/{len(demo_df)} variables have code mappings')

# Enrich politicized lifestyle variables
lifestyle_df['CodeMapping'] = lifestyle_df['VariableName'].map(mappings)
lifestyle_df.to_csv(f'{CSV_DIR}/gss_politicized_lifestyle_variables.csv', index=False)
print(f'\nSaved: {CSV_DIR}/gss_politicized_lifestyle_variables.csv')

filled_l = lifestyle_df['CodeMapping'].notna() & (lifestyle_df['CodeMapping'] != '')
print(f'  {filled_l.sum()}/{len(lifestyle_df)} variables have code mappings')

## 6. Review enriched data

In [ ]:
# Show demographic variables with mappings
print('=== DEMOGRAPHIC VARIABLES ===')
print()
for _, row in demo_df.iterrows():
    cm = row['CodeMapping'] if pd.notna(row['CodeMapping']) else 'NO MAPPING'
    print(f"{row['VariableName']:15s} | {row['Description'][:50]:50s} | {cm}")

In [ ]:
# Show politicized lifestyle variables with mappings
print('=== POLITICIZED LIFESTYLE VARIABLES ===')
print()
for _, row in lifestyle_df.iterrows():
    cm = row['CodeMapping'] if pd.notna(row['CodeMapping']) else 'NO MAPPING'
    print(f"{row['VariableName']:15s} | {row['Description'][:50]:50s} | {cm}")